In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


In [ ]:
import copy
import gc
import hashlib
import io
import json
import math
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class Config:
    data_dir: str = "datascape-2026-data-mining-competition"
    output_dir: str = "artifacts/baseline"
    classes: tuple[str, ...] = ("Aman", "Rusak", "Lainnya")
    seed: int = 42
    image_size: int = 224
    batch_size: int | None = 128
    scan_workers: int = 4
    cache_dir: str = "artifacts/data_cache"
    max_batch_size: int = 256
    gpu_memory_fraction: float = 0.8
    num_workers: int = 2 if torch.cuda.is_available() else 0
    prefetch_factor: int = 2
    mixed_precision: bool = True
    deterministic: bool = False
    show_progress: bool = True
    epochs: int = 25
    patience: int = 5
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    validation_fraction: float = 0.2

    def __post_init__(self):
        if min(self.image_size, self.epochs, self.patience, self.max_batch_size, self.prefetch_factor, self.scan_workers) < 1:
            raise ValueError("Image size, epochs, patience, maximum batch and prefetch factor must be positive.")
        if self.batch_size is not None and self.batch_size < 1:
            raise ValueError("Batch size must be positive or None for automatic selection.")
        if self.num_workers < 0:
            raise ValueError("Worker count must not be negative.")
        if not 0 < self.gpu_memory_fraction < 1:
            raise ValueError("GPU memory fraction must be between 0 and 1.")
        if self.image_size < 8:
            raise ValueError("Image size must be at least 8.")
        if not 0 < self.validation_fraction < 1:
            raise ValueError("Validation fraction must be between 0 and 1.")
        if self.learning_rate <= 0 or self.weight_decay < 0:
            raise ValueError("Invalid optimizer configuration.")
        if len(self.classes) < 2 or len(set(self.classes)) != len(self.classes):
            raise ValueError("Classes must contain at least two unique names.")


def configure_training(config: Config | None = None, **changes) -> Config:
    return replace(config if config is not None else Config(), **changes)


def set_seed(seed: int, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic


config = configure_training(
    batch_size=128,
    max_batch_size=256,
    epochs=25,
    mixed_precision=True,
    show_progress=True,
)
set_seed(config.seed, config.deterministic)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def authenticate_kaggle() -> None:
    import kagglehub
    from kagglehub.config import set_kaggle_api_token

    token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
    if not token:
        raise RuntimeError("Set KAGGLE_API_TOKEN in the environment or Colab Secrets.")
    set_kaggle_api_token(token)
    kagglehub.whoami(verbose=False)


def resolve_data_dir(config: Config) -> Path:
    root = Path(config.data_dir).expanduser().resolve()
    if not root.exists():
        import kagglehub

        authenticate_kaggle()
        root = Path(kagglehub.competition_download(
            "datascape-2026-data-mining-competition",
        )).resolve()
    required_dirs = [root / "Train_Set" / label for label in config.classes]
    required_dirs.append(root / "Test_Set")
    missing = [str(path) for path in required_dirs if not path.is_dir()]
    if not (root / "sample_submission.csv").is_file():
        missing.append(str(root / "sample_submission.csv"))
    if missing:
        raise FileNotFoundError(
            "Dataset is incomplete or data_dir points to the wrong folder: " + ", ".join(missing)
        )
    return root


def load_rgb_image(source) -> Image.Image:
    with Image.open(source) as image:
        if image.mode == "P":
            image = image.convert("RGBA")
        image = ImageOps.exif_transpose(image)
        if image.mode in {"RGBA", "LA"} or "transparency" in image.info:
            rgba = image.convert("RGBA")
            result = Image.new("RGB", rgba.size, (128, 128, 128))
            result.paste(rgba, mask=rgba.getchannel("A"))
            return result
        return image.convert("RGB")


def inspect_training_image(record):
    content = Path(record["path"]).read_bytes()
    result = dict(record, sha256=hashlib.sha256(content).hexdigest(), read_error="")
    try:
        load_rgb_image(io.BytesIO(content)).close()
    except (OSError, ValueError) as exc:
        result["read_error"] = f"{type(exc).__name__}: {exc}"
    return result


def read_training_data(
    root: Path, classes: tuple[str, ...], cache_path: Path | None = None,
    workers: int = 4, show_progress: bool = True,
) -> pd.DataFrame:
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    records = []
    for target, label in enumerate(classes):
        folder = root / "Train_Set" / label
        if not folder.is_dir():
            raise FileNotFoundError(folder)
        paths = sorted(path for path in folder.rglob("*") if path.is_file() and path.suffix.lower() in extensions)
        if not paths:
            raise ValueError(f"No images found for class: {label}")
        for path in paths:
            stat = path.stat()
            records.append({
                "path": str(path.resolve()), "target": target,
                "size": stat.st_size, "mtime_ns": stat.st_mtime_ns,
            })
    cached = {}
    if cache_path is not None and cache_path.is_file():
        try:
            manifest = json.loads(cache_path.read_text())
        except json.JSONDecodeError:
            manifest = {}
        if manifest.get("version") == 1:
            cached = manifest.get("images", {})
    completed = {}
    pending = []
    for record in records:
        previous = cached.get(record["path"], {})
        if all(previous.get(key) == value for key, value in record.items()) and "sha256" in previous and "read_error" in previous:
            completed[record["path"]] = previous
        else:
            pending.append(record)

    def save_cache():
        if cache_path is not None:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            temporary = cache_path.with_suffix(".tmp")
            temporary.write_text(json.dumps({"version": 1, "images": completed}))
            temporary.replace(cache_path)

    with tqdm(total=len(records), initial=len(completed), desc="Check training images", unit="image", disable=not show_progress) as progress:
        executor = ThreadPoolExecutor(max_workers=workers)
        try:
            for record in executor.map(inspect_training_image, pending):
                completed[record["path"]] = record
                progress.update(1)
                if len(completed) % 256 == 0:
                    save_cache()
        finally:
            executor.shutdown(wait=True, cancel_futures=True)
            save_cache()
    return pd.DataFrame([completed[record["path"]] for record in records])


def exclude_invalid_training_data(frame: pd.DataFrame):
    frame = frame.copy()
    conflicting = frame.groupby("sha256")["target"].transform("nunique").gt(1)
    frame["exclusion_reason"] = ""
    frame.loc[conflicting, "exclusion_reason"] = "conflicting_labels"
    frame.loc[frame["read_error"].ne(""), "exclusion_reason"] = "unreadable_image"
    excluded = frame[frame["exclusion_reason"].ne("")].reset_index(drop=True)
    usable = frame[frame["exclusion_reason"].eq("")].reset_index(drop=True)
    return usable.drop(columns=["read_error", "exclusion_reason"]), excluded


def split_training_data(frame: pd.DataFrame, config: Config):
    unique = frame.drop_duplicates("sha256")
    train_groups, valid_groups = train_test_split(
        unique,
        test_size=config.validation_fraction,
        stratify=unique["target"],
        random_state=config.seed,
    )
    train = frame[frame["sha256"].isin(train_groups["sha256"])].reset_index(drop=True)
    valid = frame[frame["sha256"].isin(valid_groups["sha256"])].reset_index(drop=True)
    expected = set(range(len(config.classes)))
    if set(train["target"]) != expected or set(valid["target"]) != expected:
        raise ValueError("Every class must be present in both training and validation.")
    return train, valid

In [ ]:
class ResizeWithPadding:
    def __init__(self, size: int):
        self.size = size

    def __call__(self, image: Image.Image) -> Image.Image:
        return ImageOps.pad(
            image,
            (self.size, self.size),
            method=Image.Resampling.BILINEAR,
            color=(128, 128, 128),
        )


class ImageDataset(Dataset):
    def __init__(self, paths, transform, targets=None):
        self.paths = tuple(Path(path) for path in paths)
        self.targets = None if targets is None else tuple(int(value) for value in targets)
        self.transform = transform
        if self.targets is not None and len(self.paths) != len(self.targets):
            raise ValueError("Image and target counts do not match.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        try:
            with load_rgb_image(path) as image:
                tensor = self.transform(image)
        except (OSError, ValueError) as error:
            raise RuntimeError(f"Cannot load image: {path}") from error
        if self.targets is None:
            return tensor
        return tensor, self.targets[index]


def make_transform(size: int, training: bool):
    operations = [ResizeWithPadding(size)]
    if training:
        operations.append(transforms.RandomHorizontalFlip())
    operations.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])
    return transforms.Compose(operations)


def seed_worker(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed)
    random.seed(seed)


def make_loader(dataset, config: Config, shuffle: bool):
    if config.batch_size is None:
        raise ValueError("Resolve batch size before creating data loaders.")
    options = {}
    if config.num_workers > 0:
        options.update(prefetch_factor=config.prefetch_factor, persistent_workers=True)
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=shuffle,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(config.seed),
        **options,
    )

In [ ]:
class PackageClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        layers = []
        channels = (3, 32, 64, 128)
        for input_channels, output_channels in zip(channels, channels[1:]):
            layers.extend([
                nn.Conv2d(input_channels, output_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(output_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ])
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(channels[-1], num_classes),
        )

    def forward(self, images):
        return self.classifier(self.features(images))


def train_epoch(model, loader, optimizer, criterion, device, scaler, config, epoch):
    model.train()
    total_loss = 0.0
    samples = 0
    with tqdm(
        loader, desc=f"Train {epoch}/{config.epochs}", unit="batch", leave=False,
        disable=not config.show_progress,
    ) as progress:
        for step, (images, targets) in enumerate(progress, 1):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=scaler.is_enabled()):
                loss = criterion(model(images), targets)
            if not torch.isfinite(loss):
                raise FloatingPointError("Training loss is not finite.")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            samples += targets.size(0)
            total_loss += loss.item() * targets.size(0)
            if step % 10 == 0 or step == len(loader):
                progress.set_postfix(loss=f"{total_loss / samples:.4f}", samples=samples)
    return total_loss / samples


@torch.inference_mode()
def evaluate(model, loader, criterion, device, num_classes, mixed_precision=False, show_progress=False):
    model.eval()
    total_loss = 0.0
    actual, predicted = [], []
    with tqdm(loader, desc="Validation", unit="batch", leave=False, disable=not show_progress) as progress:
        for images, targets in progress:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=mixed_precision and device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            if not torch.isfinite(loss):
                raise FloatingPointError("Validation loss is not finite.")
            total_loss += loss.item() * targets.size(0)
            actual.extend(targets.cpu().tolist())
            predicted.extend(logits.argmax(dim=1).cpu().tolist())
    metrics = {
        "loss": total_loss / len(loader.dataset),
        "macro_f1": f1_score(
            actual, predicted, labels=list(range(num_classes)), average="macro", zero_division=0,
        ),
    }
    return metrics, actual, predicted

In [ ]:
def measure_batch_memory(model, batch_size, config, device):
    probe = copy.deepcopy(model).train()
    optimizer = torch.optim.AdamW(
        probe.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=config.mixed_precision)
    images = torch.zeros(batch_size, 3, config.image_size, config.image_size, device=device)
    targets = torch.zeros(batch_size, dtype=torch.long, device=device)
    torch.cuda.reset_peak_memory_stats(device)
    for _ in range(2):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=config.mixed_precision):
            loss = nn.functional.cross_entropy(probe(images), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    torch.cuda.synchronize(device)
    return torch.cuda.max_memory_allocated(device)


def select_batch_size(model, config: Config, device, dataset_size: int) -> int:
    if config.batch_size is not None:
        return config.batch_size
    limit = min(config.max_batch_size, dataset_size)
    if device.type != "cuda":
        return min(32, limit)
    torch.cuda.empty_cache()
    free_memory, _ = torch.cuda.mem_get_info(device)
    memory_budget = torch.cuda.memory_allocated(device) + free_memory * config.gpu_memory_fraction
    candidates = [2 ** power for power in range(limit.bit_length())]
    if candidates[-1] != limit:
        candidates.append(limit)
    selected = 0
    with torch.random.fork_rng(devices=[device.index or torch.cuda.current_device()]), torch.backends.cudnn.flags(benchmark=False):
        for batch_size in tqdm(candidates, desc="Batch size", unit="trial", disable=not config.show_progress):
            try:
                peak = measure_batch_memory(model, batch_size, config, device)
            except torch.cuda.OutOfMemoryError:
                break
            finally:
                gc.collect()
                torch.cuda.empty_cache()
            if peak > memory_budget:
                break
            selected = batch_size
    if selected == 0:
        raise RuntimeError("No batch fits the GPU memory budget. Reduce image_size or free GPU memory.")
    return selected


def fit(model, train_loader, valid_loader, config: Config, device):
    output = Path(config.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output / "best.pt"
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=config.mixed_precision and device.type == "cuda")
    criterion = nn.CrossEntropyLoss()
    history = []
    best_score = -math.inf
    stale_epochs = 0
    with tqdm(range(1, config.epochs + 1), desc="Epochs", unit="epoch", disable=not config.show_progress) as progress:
        for epoch in progress:
            started = time.perf_counter()
            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats(device)
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler, config, epoch)
            metrics, _, _ = evaluate(
                model, valid_loader, criterion, device, len(config.classes),
                mixed_precision=config.mixed_precision, show_progress=config.show_progress,
            )
            epoch_seconds = time.perf_counter() - started
            peak_memory = torch.cuda.max_memory_allocated(device) / 2 ** 30 if device.type == "cuda" else 0.0
            history.append({
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": metrics["loss"],
                "valid_macro_f1": metrics["macro_f1"],
                "epoch_seconds": epoch_seconds,
                "peak_gpu_gib": peak_memory,
                "batch_size": config.batch_size,
                "learning_rate": optimizer.param_groups[0]["lr"],
            })
            pd.DataFrame(history).to_csv(output / "history.csv", index=False)
            if metrics["macro_f1"] > best_score:
                best_score = metrics["macro_f1"]
                stale_epochs = 0
                temporary_path = output / "best.tmp"
                torch.save({
                    "model_state": model.state_dict(),
                    "config": asdict(config),
                    "epoch": epoch,
                    "valid_macro_f1": best_score,
                }, temporary_path)
                temporary_path.replace(checkpoint_path)
            else:
                stale_epochs += 1
            progress.set_postfix(
                train_loss=f"{train_loss:.4f}", valid_loss=f"{metrics['loss']:.4f}",
                f1=f"{metrics['macro_f1']:.4f}", best_f1=f"{best_score:.4f}",
                patience=f"{stale_epochs}/{config.patience}", gpu_gib=f"{peak_memory:.2f}",
            )
            if stale_epochs >= config.patience:
                if config.show_progress:
                    tqdm.write(f"Early stopping at epoch {epoch}; best macro F1: {best_score:.4f}")
                break
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state"])
    return pd.DataFrame(history)


def run_training(config: Config, train_data: pd.DataFrame, valid_data: pd.DataFrame):
    set_seed(config.seed, config.deterministic)
    if config.show_progress:
        tqdm.write("Preparing training")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PackageClassifier(len(config.classes)).to(device)
    batch_size = select_batch_size(model, config, device, len(train_data))
    run_config = replace(config, batch_size=batch_size)
    output = Path(run_config.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    (output / "config.json").write_text(json.dumps(asdict(run_config), indent=2))
    train_data.to_csv(output / "train_split.csv", index=False)
    valid_data.to_csv(output / "valid_split.csv", index=False)
    train_dataset = ImageDataset(
        train_data["path"], make_transform(run_config.image_size, training=True), train_data["target"],
    )
    valid_dataset = ImageDataset(
        valid_data["path"], make_transform(run_config.image_size, training=False), valid_data["target"],
    )
    train_loader = make_loader(train_dataset, run_config, shuffle=True)
    valid_loader = make_loader(valid_dataset, run_config, shuffle=False)
    if config.show_progress:
        name = torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU"
        tqdm.write(
            f"{name} | batch_size={batch_size} | workers={run_config.num_workers} | "
            f"mixed_precision={run_config.mixed_precision and device.type == 'cuda'}"
        )
    history = fit(model, train_loader, valid_loader, run_config, device)
    return model, history, run_config

In [ ]:
root = resolve_data_dir(config)
output = Path(config.output_dir)
output.mkdir(parents=True, exist_ok=True)

training_data = read_training_data(
    root, config.classes,
    cache_path=Path(config.cache_dir) / "training_manifest.json",
    workers=config.scan_workers, show_progress=config.show_progress,
)
training_data, excluded_data = exclude_invalid_training_data(training_data)
excluded_data.assign(label=excluded_data["target"].map(dict(enumerate(config.classes)))).to_csv(
    output / "excluded_training_data.csv", index=False,
)
train_data, valid_data = split_training_data(training_data, config)
model, history, run_config = run_training(config, train_data, valid_data)
history

In [ ]:
output = Path(run_config.output_dir)
device = next(model.parameters()).device
valid_dataset = ImageDataset(
    valid_data["path"], make_transform(run_config.image_size, training=False), valid_data["target"],
)
valid_loader = make_loader(valid_dataset, run_config, shuffle=False)
validation_metrics, actual, predicted = evaluate(
    model, valid_loader, nn.CrossEntropyLoss(), device, len(run_config.classes),
    mixed_precision=run_config.mixed_precision, show_progress=run_config.show_progress,
)
report = classification_report(
    actual,
    predicted,
    labels=list(range(len(run_config.classes))),
    target_names=list(run_config.classes),
    output_dict=True,
    zero_division=0,
)
(output / "validation.json").write_text(json.dumps(report, indent=2))
validation_predictions = valid_data.assign(actual=actual, predicted=predicted)
validation_predictions.to_csv(output / "validation_predictions.csv", index=False)
pd.DataFrame(report).transpose()

In [ ]:
def read_submission_template(root: Path):
    template = pd.read_csv(root / "sample_submission.csv", dtype={"filename": str})
    if list(template.columns) != ["filename", "label"] or template.empty:
        raise ValueError("Expected a nonempty submission template with filename and label columns.")
    if template["filename"].isna().any() or template["filename"].duplicated().any():
        raise ValueError("Submission filenames must be non-null and unique.")
    paths = []
    for filename in template["filename"]:
        if Path(filename).name != filename:
            raise ValueError(f"Expected a filename without directories: {filename}")
        path = root / "Test_Set" / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        paths.append(path)
    return template, paths


@torch.inference_mode()
def predict(model, loader, device, mixed_precision=False, show_progress=False):
    model.eval()
    predictions = []
    for images in tqdm(loader, desc="Predict", unit="batch", disable=not show_progress):
        with torch.autocast(device_type=device.type, enabled=mixed_precision and device.type == "cuda"):
            logits = model(images.to(device, non_blocking=True))
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Prediction logits are not finite.")
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return predictions


def write_submission(template, predictions, classes, destination: Path):
    if len(predictions) != len(template):
        raise ValueError("Prediction count does not match submission template.")
    if any(index not in range(len(classes)) for index in predictions):
        raise ValueError("Predictions contain invalid class indices.")
    submission = template.copy()
    submission["label"] = [classes[index] for index in predictions]
    destination.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(destination, index=False)
    return submission

In [ ]:
template, test_paths = read_submission_template(root)
test_dataset = ImageDataset(test_paths, make_transform(run_config.image_size, training=False))
test_loader = make_loader(test_dataset, run_config, shuffle=False)
test_predictions = predict(
    model, test_loader, device,
    mixed_precision=run_config.mixed_precision, show_progress=run_config.show_progress,
)
submission = write_submission(template, test_predictions, run_config.classes, output / "submission.csv")
submission.head()